In [10]:
import pandas as pd
orders_labeled = pd.read_csv("artifacts/orders_labeled.csv")
print("Labeled dataset loaded successfully.")
print("Shape:", orders_labeled.shape)

Labeled dataset loaded successfully.
Shape: (96476, 22)


In [11]:
print(orders_labeled[[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "is_late"]].head())

  order_purchase_timestamp order_delivered_customer_date  \
0      2017-10-02 10:56:33           2017-10-10 21:25:13   
1      2018-07-24 20:41:37           2018-08-07 15:27:45   
2      2018-08-08 08:38:49           2018-08-17 18:06:29   
3      2017-11-18 19:28:06           2017-12-02 00:28:42   
4      2018-02-13 21:18:39           2018-02-16 18:17:02   

  order_estimated_delivery_date  is_late  
0                    2017-10-18        0  
1                    2018-08-13        0  
2                    2018-09-04        0  
3                    2017-12-15        0  
4                    2018-02-26        0  


In [12]:
orders_labeled["order_purchase_timestamp"] = pd.to_datetime(
    orders_labeled["order_purchase_timestamp"],
    errors="coerce")

orders_labeled = orders_labeled.sort_values(
    "order_purchase_timestamp").reset_index(drop=True)

In [13]:
print("Earliest order:", orders_labeled["order_purchase_timestamp"].min())
print("Latest order:", orders_labeled["order_purchase_timestamp"].max())

Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37


In [14]:
print(
    orders_labeled["is_late"]
    .value_counts()
    .rename(index={0: "On time", 1: "Late"})) 

is_late
On time    88649
Late        7827
Name: count, dtype: int64


In [15]:
# Time-based split:
n = len(orders_labeled)
train_end = int(n * 0.70)
validation_end = int(n * 0.85)
train = orders_labeled.iloc[:train_end].copy()
validation = orders_labeled.iloc[
    train_end:validation_end].copy()

test = orders_labeled.iloc[
    validation_end:].copy()
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 22)
Validation: (14471, 22)
Test: (14472, 22)


In [17]:
print("TRAIN")
print(train["order_purchase_timestamp"].min())
print(train["order_purchase_timestamp"].max())

print("\nVALIDATION")
print(validation["order_purchase_timestamp"].min())
print(validation["order_purchase_timestamp"].max())

print("\nTEST")
print(test["order_purchase_timestamp"].min())
print(test["order_purchase_timestamp"].max())

TRAIN
2016-09-15 12:16:38
2018-04-15 20:07:56

VALIDATION
2018-04-15 20:10:23
2018-06-21 07:50:39

TEST
2018-06-21 08:29:29
2018-08-29 15:00:37


In [18]:
def label_distribution(df, name):
    counts = df["is_late"].value_counts()
    percentages = df["is_late"].value_counts(normalize=True) * 100

    print(f"\n{name}")
    print("On-time:", counts.get(0, 0))
    print("Late:", counts.get(1, 0))
    print("Late %:", round(percentages.get(1, 0), 2))


label_distribution(train, "TRAIN")
label_distribution(validation, "VALIDATION")
label_distribution(test, "TEST")


TRAIN
On-time: 61436
Late: 6097
Late %: 9.03

VALIDATION
On-time: 13698
Late: 773
Late %: 5.34

TEST
On-time: 13515
Late: 957
Late %: 6.61


 `Random Stratified Split --- just for comparison ---`

In [19]:
from sklearn.model_selection import train_test_split
train_random, temp_random = train_test_split(
    orders_labeled,
    test_size=0.30,
    random_state=42,
    stratify=orders_labeled["is_late"])

validation_random, test_random = train_test_split(
    temp_random,
    test_size=0.50,
    random_state=42,
    stratify=temp_random["is_late"])

print("Random split:")
print("Train:", train_random.shape)
print("Validation:", validation_random.shape)
print("Test:", test_random.shape)

Random split:
Train: (67533, 22)
Validation: (14471, 22)
Test: (14472, 22)


## `Split Strategy`  
*`A time-based split`* *`was selected`* because the goal is to predict delivery delays using historical orders.  
The data was split chronologically into:  
*- 70% training*  
*- 15% validation*  
*- 15% test*  
This prevents future orders from influencing the training process.  
A stratified random split was also examined as a comparison. While stratification preserves the label distribution across splits, the time-based split better represents a real-world prediction scenario.  

In [20]:
train.to_csv("artifacts/train.csv",index=False)
validation.to_csv("artifacts/validation.csv",index=False)
test.to_csv("artifacts/test.csv",index=False)
print("Train, validation, and test artifacts saved successfully.")

Train, validation, and test artifacts saved successfully.
